# Agent Definition — EcoTravel Agent

This notebook defines the EcoTravel Agent architecture. The agent uses:
- **LLM**: Claude (claude-sonnet-4-6) via Anthropic SDK
- **Tools**: 4 custom tools (search_hotels, get_hotel_details, find_nearby_destinations, build_itinerary)
- **Session Memory**: Stores user preferences collected in the preference shelter and filters all hotel results
- **Tracing**: LangSmith captures all interactions for evaluation

The agent only answers travel-related questions. Non-travel requests are gracefully rejected.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, str(Path.cwd().parent))
load_dotenv(Path.cwd().parent / ".env")

from src.agent import EcoTravelAgent
from src.models import UserPreferences
from src.tools.agent_tools import TOOL_DEFINITIONS
from src.tracing import traced_chat
import json

agent = EcoTravelAgent(model="claude-sonnet-4-6")
print("Agent initialized. Model:", agent.model)
print("Tools available:", len(TOOL_DEFINITIONS))

## Tool Definitions

The agent has 4 tools. Each tool is defined as a JSON schema that Claude uses to decide when and how to call it.

In [ ]:
for tool in TOOL_DEFINITIONS:
    print(f"\n{'='*55}")
    print(f"Tool: {tool['name']}")
    print(f"Description: {tool['description'][:100]}...")
    required = tool['input_schema'].get('required', [])
    print(f"Required inputs: {required}")

## Session Memory — Preference Shelter

The preference shelter collects user criteria before any search. These preferences are stored in `SessionMemory` and applied as filters to every hotel result.

| Preference | Value | Effect |
|-----------|-------|--------|
| Budget | $175/night | Filters out hotels above this rate |
| Weather | warm | Informs destination recommendations |
| Max drive | 25 miles | Filters hotels beyond this distance |
| Crowd tolerance | low | Filters hotels with crowd score > 35 |

In [ ]:
prefs = UserPreferences(
    budget_per_night=175.0,
    weather_preference="warm",
    max_drive_miles=25,
    crowd_tolerance="low",
)
agent.memory.set_preferences(prefs)

print("Preferences set in SessionMemory:")
print(f"  Budget: ${prefs.budget_per_night:.0f}/night")
print(f"  Weather: {prefs.weather_preference}")
print(f"  Max drive: {prefs.max_drive_miles} miles")
print(f"  Crowd tolerance: {prefs.crowd_tolerance}")
print(f"  Crowd score threshold (auto): {prefs.crowd_score_threshold}")
print(f"\nPreferences active: {agent.memory.has_preferences()}")

## Example 1: Hotel Search

The agent calls the `search_hotels` tool, filters by user preferences, calculates crowd scores, and returns a sorted table.

In [ ]:
response1 = traced_chat(
    agent,
    "Find hotels in Asheville, NC from August 1-5 within 25 miles",
    run_name="demo-hotel-search"
)
print(response1)

## Example 2: Hotel Details

The `get_hotel_details` tool retrieves a summarized review and rating breakdown for the top result.

In [ ]:
response2 = traced_chat(
    agent,
    "Tell me more about the top result, including recent reviews",
    run_name="demo-hotel-details"
)
print(response2)

## Example 3: Nearby Destinations

The `find_nearby_destinations` tool uses GeoNames data to suggest quieter towns nearby.

In [ ]:
response3 = traced_chat(
    agent,
    "What are quieter nearby towns I could consider instead?",
    run_name="demo-nearby"
)
print(response3)

## Graceful Rejection 1 — Off-Topic Query

The agent only handles travel and accommodation. Any unrelated question is rejected and redirected.

In [ ]:
rejection1 = traced_chat(
    agent,
    "What is the capital of France?",
    run_name="demo-rejection-1"
)
print(rejection1)
print()

# Verify rejection redirects to travel topics
is_rejected = any(
    kw in rejection1.lower()
    for kw in ["travel", "hotel", "destination", "accommodation"]
)
print("Rejection correctly redirects to travel topics:", is_rejected)
assert is_rejected, "Expected rejection message to mention travel topics"

## Graceful Rejection 2 — Unrelated Creative Request

A second rejection example to confirm consistent behavior across different off-topic request types.

In [ ]:
rejection2 = traced_chat(
    agent,
    "Write me a poem about mountains",
    run_name="demo-rejection-2"
)
print(rejection2)
print()

is_rejected2 = any(
    kw in rejection2.lower()
    for kw in ["travel", "hotel", "destination", "accommodation"]
)
print("Rejection correctly redirects to travel topics:", is_rejected2)
assert is_rejected2, "Expected rejection message to mention travel topics"

## Session Memory State

SessionMemory persists user preferences and conversation history across the entire session.

In [ ]:
prefs = agent.memory.get_preferences()
history = agent.memory.get_conversation_history()

print(f"Preferences stored: {prefs is not None}")
print(f"Conversation turns in memory: {len(history)}")
print(f"\nConversation preview (last 3 turns):")
for msg in history[-3:]:
    role = msg['role']
    content = str(msg['content'])[:80]
    print(f"  [{role}]: {content}...")

print(f"\nTool count: {len(TOOL_DEFINITIONS)}")
print("Agent is ready for deployment.")